In [2]:
#Merging data into one folder

import shutil
import os
from pathlib import Path

#Merge datasets
SOURCE_DIRS = {
    "Tumour" : ["C:\\Users\\USER\\Downloads\\CodeDataS\\Mri_clean_tumor_512x512"],
    "Stroke" : ["C:\\Users\\USER\\Downloads\\CodeDataS\\Mri_clean_Stroke_512x512"],
    "Normal" : ["C:\\Users\\USER\\Downloads\\CodeDataS\\Mri_clean_normal_512x512"],
}
DEST = Path("brain_mri_combined")

for label, sources in SOURCE_DIRS.items(): 
    dest_class = DEST / label
    dest_class.mkdir(parents = True, exist_ok = True)
    for src in sources: 
        for f in Path(src).rglob("*"):
            if f.suffix.lower() in {".jpg"}:

                #avoid name collisions
                new_name = f"{Path(src).name}_{f.name}"
                shutil.copy2(f, dest_class / new_name)

In [3]:
#Data Split
import tensorflow as tf

DATA_DIR = "brain_mri_combined"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

#Load full dataset with 70% training and 30% temporary validation set
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split = 0.3, 
    subset = "training", 
    seed = SEED,
    image_size = IMG_SIZE,
    batch_size = BATCH_SIZE
)
val_temp_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split = 0.3,
    subset = "validation",
    seed = SEED,
    image_size = IMG_SIZE,
    batch_size = BATCH_SIZE
)

#Further split the 30% temp set into validation and test (50/50)
val_batches = tf.data.experimental.cardinality(val_temp_ds)
test_ds = val_temp_ds.take(val_batches // 2)
val_ds = val_temp_ds.skip(val_batches // 2)

print(f"Class names: {train_ds.class_names}")
print(f"Training batches: {tf.data.experimental.cardinality(train_ds)}")
print(f"Validation batches: {tf.data.experimental.cardinality(val_ds)}")
print(f"Test batches: {tf.data.experimental.cardinality(test_ds)}")

Found 3308 files belonging to 3 classes.
Using 2316 files for training.
Found 3308 files belonging to 3 classes.
Using 992 files for validation.
Class names: ['Normal', 'Stroke', 'Tumour']
Training batches: 73
Validation batches: 16
Test batches: 15


In [5]:
#Preprocess
AUTOTUNE = tf.data.AUTOTUNE

def preprocess(image, label):
    # Convert grayscale to RGB (replicate channels)
    #image = tf.image.grayscale_to_rgb(image)
    return image, label

train_ds = train_ds.map(preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds = val_ds.map(preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds = test_ds.map(preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

In [6]:
#Augmentation
augment_layers = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(factor = 0.1)
])
def preprocess_with_augmentation(image, label):
    image = tf.image.random_brightness(image, 0.1)
    return image, label

In [7]:
#EfficientNetB0 Model
from tensorflow.keras import layers, models, optimizers

base_model = tf.keras.applications.EfficientNetB0(
    include_top = False, 
    weights = "imagenet",
    input_shape = (224, 224, 3),
    pooling = "avg"
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.Dropout(0.2),
    layers.Dense(3, activation="softmax")
])

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [8]:
#Class Imbalance:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

train_labels = np.concatenate([y for x, y in train_ds], axis=0)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels
)
class_weight_dict = dict(enumerate(class_weights))

In [9]:
#Train with Two-Stage Transfer Learning
#Stage 1: Feature extraction (backbone frozen)
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weight_dict,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=3, restore_best_weights=True
    )]
)
#Stage 2: Fine-tuning (Unfreeze backbone)
base_model.trainable = True
model.compile(
    optimizer=optimizers.Adam(learning_rate=2e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weight_dict,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=3, restore_best_weights=True
    )]
)

Epoch 1/10


C:\Users\USER\anaconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


73/73 ━━━━━━━━━━━━━━━━━━━━ 62s 642ms/step - accuracy: 0.7215 - loss: 0.6660 - val_accuracy: 0.8145 - val_loss: 0.4702
Epoch 2/10
73/73 ━━━━━━━━━━━━━━━━━━━━ 46s 624ms/step - accuracy: 0.8143 - loss: 0.4608 - val_accuracy: 0.8516 - val_loss: 0.4093
Epoch 3/10
73/73 ━━━━━━━━━━━━━━━━━━━━ 46s 628ms/step - accuracy: 0.8325 - loss: 0.4058 - val_accuracy: 0.8477 - val_loss: 0.3839
Epoch 4/10
73/73 ━━━━━━━━━━━━━━━━━━━━ 45s 622ms/step - accuracy: 0.8523 - loss: 0.3679 - val_accuracy: 0.8555 - val_loss: 0.3519
Epoch 5/10
73/73 ━━━━━━━━━━━━━━━━━━━━ 46s 627ms/step - accuracy: 0.8592 - loss: 0.3505 - val_accuracy: 0.8594 - val_loss: 0.3370
Epoch 6/10
73/73 ━━━━━━━━━━━━━━━━━━━━ 45s 620ms/step - accuracy: 0.8636 - loss: 0.3415 - val_accuracy: 0.8652 - val_loss: 0.3548
Epoch 7/10
73/73 ━━━━━━━━━━━━━━━━━━━━ 48s 652ms/step - accuracy: 0.8744 - loss: 0.3220 - val_accuracy: 0.8789 - val_loss: 0.3204
Epoch 8/10
73/73 ━━━━━━━━━━━━━━━━━━━━ 48s 656ms/step - accuracy: 0.8800 - loss: 0.3084 - val_accuracy: 0.865

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Standard model evaluation directly on the dataset
test_loss, test_acc = model.evaluate(test_ds, verbose=1)
print(f"\nTrue Model.evaluate Test Accuracy: {test_acc * 100:.2f}%\n")

# Collect actuals and predictions in a single pass to ensure correct alignment
all_y_true = []
all_y_pred = []

for images, labels in test_ds:
    preds = model.predict_on_batch(images)
    pred_classes = np.argmax(preds, axis=1)

    all_y_true.extend(labels.numpy())
    all_y_pred.extend(pred_classes)

y_true_correct = np.array(all_y_true)
y_pred_correct = np.array(all_y_pred)

# Recalculate metrics
correct_accuracy = accuracy_score(y_true_correct, y_pred_correct)
print(f"Aligned Test Accuracy: {correct_accuracy * 100:.2f}%")

# Re-plot confusion matrix
cm_correct = confusion_matrix(y_true_correct, y_pred_correct)
plt.figure(figsize=(7, 5))
sns.heatmap(
    cm_correct, annot=True, fmt="d", cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Corrected Confusion Matrix (Single-Pass)")
plt.tight_layout()
plt.show()

print("\nCorrected Classification Report:")
print(classification_report(y_true_correct, y_pred_correct, target_names=class_names))

NameError: name 'model' is not defined

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# 1. Gather all true labels and predicted probabilities
all_y_true = []
all_y_probs = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    all_y_true.extend(labels.numpy())
    all_y_probs.extend(preds)

y_true_array = np.array(all_y_true)
y_probs_array = np.array(all_y_probs)

# 2. Binarize class labels for multi-class ROC calculation
n_classes = len(class_names)
y_true_bin = label_binarize(y_true_array, classes=[0, 1, 2])

# 3. Compute ROC curve and ROC area for each class
fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_probs_array[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# 4. Plot the ROC Curves
plt.figure(figsize=(8, 6))
colors = ['blue', 'red', 'green']
for i, color in zip(range(n_classes), colors):
    plt.plot(
        fpr[i], tpr[i], color=color, lw=2,
        label=f'ROC curve of class {class_names[i]} (area = {roc_auc[i]:.2f})'
    )

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
import cv2
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    # Since top_activation is nested inside the base model, we find the actual layer inside the sub-model if needed
    try:
        target_layer = model.get_layer(last_conv_layer_name)
        parent_model = model
    except ValueError:
        # Fallback to the nested base model ('efficientnetb0')
        parent_model = model.get_layer('efficientnetb0')
        target_layer = parent_model.get_layer(last_conv_layer_name)

    # Create a model that maps the inputs of the base model to the activation of its last conv layer
    grad_model = tf.keras.models.Model(
        parent_model.inputs,
        [target_layer.output, parent_model.output]
    )

    # Run gradients through the base model first, then the remaining Sequential parts
    with tf.GradientTape() as tape:
        # Forward pass through the base model first
        conv_outputs, base_outputs = grad_model(img_array)
        
        # Forward pass the base model's outputs through the rest of sequential model
        x = model.get_layer('dropout')(base_outputs)
        preds = model.get_layer('dense')(x)
        
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    # Compute the gradient of the top class with respect to the output feature map
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Generate heatmap
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # Normalize heatmap
    heatmap = tf.maximum(heatmap, 0) / tf.reduce_max(heatmap)
    return heatmap.numpy()

def save_and_display_gradcam(img, heatmap, alpha=0.4):
    # Rescale heatmap to a range 0-255
    heatmap = np.uint8(255 * heatmap)

    # Use jet colormap to colorize heatmap
    jet = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    jet = cv2.cvtColor(jet, cv2.COLOR_BGR2RGB)

    # Resize heatmap to match original image size
    jet = cv2.resize(jet, (img.shape[1], img.shape[0]))

    # Superimpose the heatmap on original image
    superimposed_img = jet * alpha + img * (1 - alpha)
    superimposed_img = np.clip(superimposed_img, 0, 255).astype(np.uint8)
    return superimposed_img

# 1. Fetch a single batch of images from the test dataset
images, labels = next(iter(test_ds))

# 2. Get the last convolutional layer of EfficientNetB0
last_conv_layer_name = "top_activation"

# 3. Choose a few sample indices to display Grad-CAM
sample_indices = [0, 1, 2]

plt.figure(figsize=(12, 10))
for idx, sample_idx in enumerate(sample_indices):
    img = images[sample_idx].numpy()
    label = labels[sample_idx].numpy()

    # Add batch dimension
    img_batch = np.expand_dims(img, axis=0)

    # Generate class predictions
    preds = model.predict(img_batch, verbose=0)
    pred_class = np.argmax(preds[0])

    # Generate heatmap
    heatmap = make_gradcam_heatmap(img_batch, model, last_conv_layer_name, pred_index=pred_class)

    # Create superimposed visualization
    gradcam_img = save_and_display_gradcam(img, heatmap)

    # Plot Original
    plt.subplot(len(sample_indices), 2, 2 * idx + 1)
    plt.imshow(img.astype(np.uint8))
    plt.title(f"Original (Actual: {class_names[label]})")
    plt.axis("off")

    # Plot Grad-CAM overlay
    plt.subplot(len(sample_indices), 2, 2 * idx + 2)
    plt.imshow(gradcam_img)
    plt.title(f"Grad-CAM (Predicted: {class_names[pred_class]} | Conf: {preds[0][pred_class]:.2f})")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Save the trained model to best_model.keras
model.save('best_model.keras')
print("Model successfully saved to 'best_model.keras'")